# Transformers & Attention: BERT Fine-tuning & Custom Implementation

**Module 4 - Notebook 4** | Skillsoft Track 2: DL Programmer

## Objectives
- Implement transformer from scratch
- BERT fine-tuning for classification
- Attention visualization
- Positional encodings & masking

In [ ]:
# Setup
!pip install transformers==4.36.0 datasets==2.16.0 -q 2>/dev/null || pip install transformers datasets -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    BertConfig,
    BertModel,
)
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Transformer from Scratch

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # Linear projections
        q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        return self.w_o(context), attn

class PositionwiseFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn_weights

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=4, num_layers=4, d_ff=1024,
                 max_len=512, num_classes=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x = self.embedding(x) * np.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        attn_weights = []
        for layer in self.layers:
            x, attn = layer(x, mask)
            attn_weights.append(attn)
        
        # Use [CLS] token (first token) for classification
        x = x[:, 0]
        return self.classifier(x), attn_weights

# Test
model = TransformerEncoder(vocab_size=1000).to(device)
x = torch.randint(0, 1000, (2, 16)).to(device)
logits, attn = model(x)
    print(f\"Output shape: {logits.shape}\")\n",print(f"Attention layers: {len(attn)}, shape per layer: {attn[0].shape}")

## 2. BERT Fine-tuning for Classification

In [ ]:
# Load dataset
dataset = load_dataset('imdb')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

tokenized = dataset.map(tokenize_function, batched=True)
tokenized = tokenized.remove_columns(['text'])
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch')

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
).to(device)

# Training arguments
training_args = TrainingArguments(
    output_dir='./bert-imdb',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
)

def compute_metrics(eval_pred):
    from sklearn.metrics import accuracy_score, f1_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'].shuffle(seed=42).select(range(5000)),
    eval_dataset=tokenized['test'].select(range(1000)),
    compute_metrics=compute_metrics,
)

# Train (commented out for demo - takes time)
# trainer.train()
# trainer.save_model('./bert-imdb-final')

print("BERT fine-tuning setup complete. Run trainer.train() to train.")

## 3. Attention Visualization

In [ ]:
# Attention visualization with BERT
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True).to(device)
    model.eval()\n",
text = "The cat sat on the mat because it was tired."
inputs = tokenizer(text, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model(**inputs)

attentions = outputs.attentions  # Tuple of (batch, heads, seq, seq)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Visualize attention for each layer
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for layer_idx in range(12):
    attn = attentions[layer_idx][0].cpu().numpy()  # (heads, seq, seq)
    # Average over heads
    attn_avg = attn.mean(axis=0)
    
    im = axes[layer_idx].imshow(attn_avg, cmap='Blues')
    axes[layer_idx].set_xticks(range(len(tokens)))
    axes[layer_idx].set_yticks(range(len(tokens)))
    axes[layer_idx].set_xticklabels(tokens, rotation=90, fontsize=6)
    axes[layer_idx].set_yticklabels(tokens, fontsize=6)
    axes[layer_idx].set_title(f'Layer {layer_idx+1}')

plt.tight_layout()
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
plt.show()

## 4. Masking Strategies

In [ ]:
# Masking visualization
def create_causal_mask(seq_len):
    """Causal mask for decoder (GPT-style)."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, seq)

def create_padding_mask(input_ids, pad_token_id=0):
    """Padding mask for encoder."""
    return (input_ids != pad_token_id).unsqueeze(1).unsqueeze(2)

def create_look_ahead_mask(seq_len):
    """Combined causal + padding mask."""
    causal = create_causal_mask(seq_len)
    return causal

# Visualize masks
seq_len = 10
causal_mask = create_causal_mask(seq_len)[0, 0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(causal_mask, cmap='RdBu')
axes[0].set_title('Causal Mask (Decoder)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# Random padding mask
pad_mask = (torch.randint(0, 2, (1, 1, 1, seq_len)) > 0).float()
axes[1].imshow(pad_mask[0, 0], cmap='RdBu')
axes[1].set_title('Padding Mask (Encoder)')

plt.tight_layout()
plt.show()

print("Masks created. Causal mask prevents attending to future tokens.")